In [23]:
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

In [24]:
PROJECT_ROOT = Path("..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [25]:
X_train = pd.read_csv(PROCESSED_DIR / "X_train_raw_baseline.csv")

y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze("columns")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (54533, 11)
y_train: (54533,)


In [26]:
NUMERICAL_FEATURES = ["age_years", "height", "weight", "ap_hi", "ap_lo"]

CATEGORICAL_FEATURES = ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]

In [ ]:
logistic_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERICAL_FEATURES),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            CATEGORICAL_FEATURES,
        ),
    ]
)

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", logistic_preprocessor),
        (
            "classifier",
            LogisticRegression(max_iter=2000, l1_ratio=0, random_state=42),
        ),
    ]
)

In [28]:
logistic_param_grid = {
    "classifier__C": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
}

In [29]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [30]:
logistic_search = RandomizedSearchCV(
    estimator=logistic_pipeline,
    param_distributions=logistic_param_grid,
    n_iter=8,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    return_train_score=False,
)

In [31]:
print("Tuning Logistic Regression...")

logistic_search.fit(X_train, y_train)

print("Finished.")

Tuning Logistic Regression...


Finished.


In [32]:
print("Best parameters:")
print(logistic_search.best_params_)

print("\nBest CV ROC-AUC:")
print(round(logistic_search.best_score_, 4))

Best parameters:
{'classifier__C': 10.0}

Best CV ROC-AUC:
0.7905


In [33]:
logistic_results = pd.DataFrame(logistic_search.cv_results_)

logistic_results[
    ["param_classifier__C", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score")

,param_classifier__C,mean_test_score,std_test_score,rank_test_score
7,10.00,0.790519,0.003580,1
6,5.00,0.790519,0.003580,2
5,2.00,0.790518,0.003580,3
4,1.00,0.790517,0.003581,4
3,0.50,0.790516,0.003582,5
2,0.10,0.790514,0.003581,6
1,0.05,0.790507,0.003582,7
0,0.01,0.790385,0.003582,8


In [34]:
hist_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", NUMERICAL_FEATURES),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            CATEGORICAL_FEATURES,
        ),
    ]
)

hist_pipeline = Pipeline(
    steps=[
        ("preprocessor", hist_preprocessor),
        (
            "classifier",
            HistGradientBoostingClassifier(random_state=42),
        ),
    ]
)

In [35]:
hist_param_grid = {
    "classifier__learning_rate": [0.03, 0.05, 0.08, 0.1],
    "classifier__max_iter": [100, 150, 200, 300],
    "classifier__max_leaf_nodes": [15, 31, 63],
    "classifier__l2_regularization": [0.0, 0.1, 1.0],
}

In [36]:
hist_search = RandomizedSearchCV(
    estimator=hist_pipeline,
    param_distributions=hist_param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    return_train_score=False,
)

In [37]:
print("Tuning HistGradientBoosting...")

hist_search.fit(
    X_train,
    y_train,
)

print("Finished.")

Tuning HistGradientBoosting...
Finished.


In [38]:
print("Best parameters:")
print(hist_search.best_params_)

print("\nBest CV ROC-AUC:")
print(round(hist_search.best_score_, 4))

Best parameters:
{'classifier__max_leaf_nodes': 15, 'classifier__max_iter': 200, 'classifier__learning_rate': 0.05, 'classifier__l2_regularization': 0.1}

Best CV ROC-AUC:
0.8


In [39]:
hist_results = pd.DataFrame(hist_search.cv_results_)

hist_results[
    [
        "param_classifier__learning_rate",
        "param_classifier__max_iter",
        "param_classifier__max_leaf_nodes",
        "param_classifier__l2_regularization",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
].sort_values("rank_test_score").head(10)

,param_classifier__learning_rate,param_classifier__max_iter,param_classifier__max_leaf_nodes,param_classifier__l2_regularization,mean_test_score,std_test_score,rank_test_score
8,0.05,200,15,0.1,0.800038,0.003794,1
17,0.05,300,15,0.1,0.800038,0.003794,1
0,0.05,300,15,1.0,0.799995,0.003835,3
6,0.10,100,15,1.0,0.799976,0.003767,4
16,0.08,150,15,0.0,0.799966,0.003930,5
11,0.08,200,15,0.1,0.799949,0.003771,6
9,0.05,200,15,0.0,0.799934,0.003818,7
5,0.05,100,15,0.0,0.799918,0.003838,8
14,0.03,150,31,1.0,0.799912,0.003768,9
15,0.05,150,31,0.1,0.799864,0.003780,10


In [40]:
tuning_comparison = pd.DataFrame(
    [
        {
            "model": "Logistic Regression",
            "best_cv_roc_auc": logistic_search.best_score_,
        },
        {"model": "HistGradientBoosting", "best_cv_roc_auc": hist_search.best_score_},
    ]
)

tuning_comparison["best_cv_roc_auc"] = tuning_comparison["best_cv_roc_auc"].round(4)

tuning_comparison

,model,best_cv_roc_auc
0,Logistic Regression,0.7905
1,HistGradientBoosting,0.8000


In [41]:
best_logistic_model = logistic_search.best_estimator_

best_hist_model = hist_search.best_estimator_

In [42]:
joblib.dump(best_logistic_model, MODELS_DIR / "logistic_regression_tuned.joblib")

joblib.dump(best_hist_model, MODELS_DIR / "histgradientboosting_tuned.joblib")

['../models/histgradientboosting_tuned.joblib']

In [43]:
logistic_results.to_csv(REPORTS_DIR / "logistic_tuning_results.csv", index=False)

hist_results.to_csv(
    REPORTS_DIR / "histgradientboosting_tuning_results.csv", index=False
)

tuning_comparison.to_csv(REPORTS_DIR / "tuning_comparison.csv", index=False)

In [44]:
summary = {
    "logistic_regression": {
        "best_params": logistic_search.best_params_,
        "best_cv_roc_auc": float(logistic_search.best_score_),
    },
    "histgradientboosting": {
        "best_params": hist_search.best_params_,
        "best_cv_roc_auc": float(hist_search.best_score_),
    },
}

with open(REPORTS_DIR / "tabular_tuning_summary.json", "w") as f:
    json.dump(summary, f, indent=4)